# Dental OPG YOLOv8m with LocaQ Loss

Run the configuration cell first. Set `DATA_YAML` to your dataset YAML and choose `OUTPUT_ROOT`. GPU 0 is the original experiment device. To evaluate an existing checkpoint, set `CHECKPOINT_OVERRIDE` and skip the training cell. Start Jupyter from the repository root.

Any saved output reflects the original run and may contain its original paths.


In [ ]:
from pathlib import Path

# Set this to the YAML for the corresponding paper dataset and splits.
DATA_YAML = str(Path("datasets/dental_opg/data.yaml").expanduser().resolve())
OUTPUT_ROOT = Path("runs/dental_opg_yolov8m_locaq").expanduser().resolve()

# Optional: point this at an existing checkpoint to run evaluation only.
CHECKPOINT_OVERRIDE = None
RUN_NAME = "LocaQLoss_YOLOv8m"

PROJECT_DIR = str(OUTPUT_ROOT / "training")
MODEL_PATH = (
    str(Path(CHECKPOINT_OVERRIDE).expanduser().resolve())
    if CHECKPOINT_OVERRIDE is not None
    else str(Path(PROJECT_DIR) / RUN_NAME / "weights" / "best.pt")
)


In [ ]:
import inspect
from locaq_loss import LocaQLoss

source = inspect.getsource(LocaQLoss)
assert "_scale_norm" not in source, "BUG: _scale_norm found. File is wrong."
print("locaq_loss.py verified clean.")

In [ ]:
# ============================================================
# LocaQLoss YOLOv8m TRAINING — Dental OPG
# Complete script — just run this
# ============================================================

import types
import sys
import os
import random
import numpy as np
import torch
from ultralytics import YOLO
from locaq_loss import LocaQLoss

# ============================================================
# VERIFY CORRECT locaq_loss.py IS LOADED
# ============================================================

import inspect
source = inspect.getsource(LocaQLoss)
if "get_assigned_targets_and_loss" in source:
    print("Correct locaq_loss.py loaded (8.4.x compatible).")
else:
    raise RuntimeError("OLD locaq_loss.py loaded. Replace the file and restart kernel.")

# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"]          = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic    = True
torch.backends.cudnn.benchmark        = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32       = False
torch.use_deterministic_algorithms(True)

# ============================================================
# PATHS
# ============================================================


# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO("yolov8m.yaml")

# ============================================================
# INJECT LocaQLoss VIA on_train_start
# Overrides trainer.model.init_criterion AFTER trainer
# creates its own model — this is the correct timing
# ============================================================

def inject_locaq_loss(trainer):
    trainer.model.init_criterion = types.MethodType(
        lambda self: LocaQLoss(
            self,
            beta=1.0,
            vfl_alpha=0.75,
            vfl_gamma=2.0,
            anchor_area=1024.0,
            fn_clamp_max=3.0
        ),
        trainer.model
    )
    print(f"[LocaQLoss] Injected into trainer.model successfully.")

model.add_callback("on_train_start", inject_locaq_loss)

# ============================================================
# TRAIN
# ============================================================

model.train(
    data=DATA_YAML,
    epochs=250,
    imgsz=640,
    batch=8,
    optimizer="AdamW",
    lr0=0.0002,
    lrf=0.01,
    weight_decay=0.01,
    cos_lr=True,
    warmup_epochs=5,
    hsv_h=0.01,
    hsv_s=0.35,
    hsv_v=0.25,
    degrees=10,
    translate=0.05,
    scale=0.25,
    shear=1.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.4,
    close_mosaic=20,
    mixup=0.0,
    seed=SEED,
    deterministic=True,
    workers=0,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    save_period=10,
    device=0
)

In [ ]:
# ============================================================
# COMPLETE TEST EVALUATION PIPELINE
# TEST RESULTS
# ============================================================

import os
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ultralytics import YOLO

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay
)

# ============================================================
# PATHS
# ============================================================



SAVE_DIR = str(OUTPUT_ROOT / "evaluation_1")

os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

# ============================================================
# RUN TEST VALIDATION
# ============================================================

metrics = model.val(
    data=DATA_YAML,
    split="test",

    imgsz=640,
    batch=8,

    conf=0.25,
    iou=0.5,

    device=0,

    save_json=True,
    plots=True,

    project=SAVE_DIR,
    name="test_eval"
)

# ============================================================
# CLASS NAMES
# ============================================================

class_names = model.names

num_classes = len(class_names)

print("\nClasses:")
print(class_names)

# ============================================================
# YOLO METRICS
# ============================================================

precision_per_class = metrics.box.p
recall_per_class = metrics.box.r

map50_per_class = metrics.box.ap50
map5095_per_class = metrics.box.ap

# ============================================================
# F1 SCORE
# ============================================================

f1_per_class = (
    2
    * precision_per_class
    * recall_per_class
) / (
    precision_per_class
    + recall_per_class
    + 1e-9
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = metrics.confusion_matrix.matrix

cm = cm[:num_classes, :num_classes]

# ============================================================
# SENSITIVITY / SPECIFICITY
# ============================================================

sensitivity_list = []
specificity_list = []

tp_list = []
fp_list = []
fn_list = []
tn_list = []

for i in range(num_classes):

    TP = cm[i, i]

    FN = cm[i, :].sum() - TP

    FP = cm[:, i].sum() - TP

    TN = cm.sum() - (TP + FN + FP)

    sensitivity = TP / (TP + FN + 1e-9)

    specificity = TN / (TN + FP + 1e-9)

    tp_list.append(TP)
    fp_list.append(FP)
    fn_list.append(FN)
    tn_list.append(TN)

    sensitivity_list.append(sensitivity)
    specificity_list.append(specificity)

# ============================================================
# MACRO METRICS
# ============================================================

macro_precision = np.mean(precision_per_class)
macro_recall = np.mean(recall_per_class)
macro_f1 = np.mean(f1_per_class)

macro_sensitivity = np.mean(sensitivity_list)
macro_specificity = np.mean(specificity_list)

macro_map50 = np.mean(map50_per_class)
macro_map5095 = np.mean(map5095_per_class)

# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame({

    "Class": [
        class_names[i]
        for i in range(num_classes)
    ],

    "Precision": precision_per_class,
    "Recall": recall_per_class,
    "F1-Score": f1_per_class,

    "Sensitivity": sensitivity_list,
    "Specificity": specificity_list,

    "mAP50": map50_per_class,
    "mAP50-95": map5095_per_class,

    "TP": tp_list,
    "FP": fp_list,
    "FN": fn_list,
    "TN": tn_list
})

results_df = results_df.round(4)

# ============================================================
# ADD MACRO ROW
# ============================================================

macro_row = pd.DataFrame([{

    "Class": "Macro Average",

    "Precision": round(macro_precision, 4),
    "Recall": round(macro_recall, 4),
    "F1-Score": round(macro_f1, 4),

    "Sensitivity": round(macro_sensitivity, 4),
    "Specificity": round(macro_specificity, 4),

    "mAP50": round(macro_map50, 4),
    "mAP50-95": round(macro_map5095, 4),

    "TP": np.sum(tp_list),
    "FP": np.sum(fp_list),
    "FN": np.sum(fn_list),
    "TN": np.sum(tn_list)
}])

results_df = pd.concat(
    [results_df, macro_row],
    ignore_index=True
)

# ============================================================
# SAVE CSV
# ============================================================

csv_path = os.path.join(
    SAVE_DIR,
    "classwise_metrics.csv"
)

results_df.to_csv(
    csv_path,
    index=False
)

print("\nSaved metrics CSV:")
print(csv_path)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n================================================")
print("TEST RESULTS")
print("================================================\n")

print(results_df)

# ============================================================
# CONFUSION MATRIX PLOT
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        class_names[i]
        for i in range(num_classes)
    ]
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

plt.title("Detection Confusion Matrix")

cm_path = os.path.join(
    SAVE_DIR,
    "confusion_matrix.png"
)

plt.savefig(
    cm_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved confusion matrix.")

# ============================================================
# CLASS-WISE BAR PLOT
# ============================================================

metrics_to_plot = [
    "Precision",
    "Recall",
    "F1-Score",
    "Sensitivity",
    "Specificity",
    "mAP50",
    "mAP50-95"
]

for metric_name in metrics_to_plot:

    plt.figure(figsize=(12, 6))

    plt.bar(
        results_df["Class"][:-1],
        results_df[metric_name][:-1]
    )

    plt.xticks(rotation=30)

    plt.ylabel(metric_name)

    plt.title(f"Class-wise {metric_name}")

    save_path = os.path.join(
        SAVE_DIR,
        f"{metric_name}.png"
    )

    plt.savefig(
        save_path,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close()

print("\nSaved class-wise metric plots.")

# ============================================================
# FALSE NEGATIVE ANALYSIS PLOT
# ============================================================

plt.figure(figsize=(12, 6))

plt.bar(
    results_df["Class"][:-1],
    results_df["FN"][:-1]
)

plt.xticks(rotation=30)

plt.ylabel("False Negatives")

plt.title("Class-wise False Negatives")

fn_path = os.path.join(
    SAVE_DIR,
    "false_negative_analysis.png"
)

plt.savefig(
    fn_path,
    dpi=600,
    bbox_inches="tight"
)

plt.close()

print("\nSaved FN analysis plot.")

# ============================================================
# MACRO SUMMARY TXT
# ============================================================

summary_path = os.path.join(
    SAVE_DIR,
    "macro_results.txt"
)

with open(summary_path, "w") as f:

    f.write("================================================\n")
    f.write("Macro Results\n")
    f.write("================================================\n\n")

    f.write(f"Precision     : {macro_precision:.4f}\n")
    f.write(f"Recall        : {macro_recall:.4f}\n")
    f.write(f"F1-Score      : {macro_f1:.4f}\n")
    f.write(f"Sensitivity   : {macro_sensitivity:.4f}\n")
    f.write(f"Specificity   : {macro_specificity:.4f}\n")
    f.write(f"mAP50         : {macro_map50:.4f}\n")
    f.write(f"mAP50-95      : {macro_map5095:.4f}\n")

print("\nSaved macro summary.")

# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n================================================")
print("FINAL MACRO RESULTS")
print("================================================")

print(f"Precision     : {macro_precision:.4f}")
print(f"Recall        : {macro_recall:.4f}")
print(f"F1-Score      : {macro_f1:.4f}")
print(f"Sensitivity   : {macro_sensitivity:.4f}")
print(f"Specificity   : {macro_specificity:.4f}")
print(f"mAP50         : {macro_map50:.4f}")
print(f"mAP50-95      : {macro_map5095:.4f}")

print("\n================================================")
print("ALL RESULTS SAVED")
print("================================================")